# Compare Global, Local, and Future Federated Models

This notebook creates paper-ready comparisons between the global MLP and client-local MLPs. It is structured so federated-learning results can be added later by dropping summary CSV files into `models/`.

The main outputs are saved under `reports/model_comparison/` as CSV, LaTeX tables, and figures.

## Common Comparison Techniques

For an imbalanced binary classification project paper, these are easy to read and easy to integrate into LaTeX:

- **Main metric table**: ROC-AUC, PR-AUC, F1, precision, recall, and accuracy for each model family.
- **Weighted local average**: aggregate client-local models by test-set size, so larger clients contribute proportionally.
- **Per-client paired table**: evaluate the global model on each client test split, then compare against that client's local model.
- **Delta table**: report `local - global` per client. Positive values mean the local model performed better on that metric.
- **Bar plots**: grouped bars for ROC-AUC and F1 are readable in papers and presentations.
- **Mean plus standard deviation**: useful when summarizing variability across clients.

For this dataset, PR-AUC, F1, precision, and recall are especially important because default cases are the minority class. Accuracy should be reported, but not used as the main success criterion.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

import torch

repo_root = Path.cwd()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.dataset.credit_data import DEFAULT_INPUT_PATH, DEFAULT_TARGET_COL, add_features, prepare_feature_sets, read_credit_dataset
from src.training.credit_mlp import DEFAULT_SEED, build_mlp_from_artifact, make_loader, predict_proba

MODELS_DIR = repo_root / "models"
CLIENT_DATA_DIR = repo_root / "data" / "IID_cl3_age_unbalanced_engineered"
GLOBAL_MODEL_PATH = MODELS_DIR / "credit_default_mlp.pt"
LOCAL_SUMMARY_PATH = MODELS_DIR / CLIENT_DATA_DIR.name / "client_model_summary.csv"
OUTPUT_DIR = repo_root / "reports" / "model_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Add explicit future paths here if your federated code writes a different filename.
FUTURE_FEDERATED_SUMMARY_PATHS = [
    # MODELS_DIR / "federated" / "model_summary.csv",
]
AUTO_FEDERATED_SUMMARY_PATHS = sorted(MODELS_DIR.glob("**/federated*_summary.csv"))
FEDERATED_SUMMARY_PATHS = sorted(set(FUTURE_FEDERATED_SUMMARY_PATHS + AUTO_FEDERATED_SUMMARY_PATHS))

SEED = DEFAULT_SEED
TARGET_COL = DEFAULT_TARGET_COL
METRIC_COLS = [
    "test_roc_auc",
    "test_pr_auc",
    "test_f1",
    "test_precision",
    "test_recall",
    "test_accuracy",
]
METRIC_LABELS = {
    "test_roc_auc": "ROC-AUC",
    "test_pr_auc": "PR-AUC",
    "test_f1": "F1",
    "test_precision": "Precision",
    "test_recall": "Recall",
    "test_accuracy": "Accuracy",
}

print("Global model:", GLOBAL_MODEL_PATH)
print("Local summary:", LOCAL_SUMMARY_PATH)
print("Client data:", CLIENT_DATA_DIR)
print("Output folder:", OUTPUT_DIR)
print("Federated summaries found:", FEDERATED_SUMMARY_PATHS)


In [ ]:
def load_torch_artifact(path):
    path = Path(path)
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def classification_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "test_roc_auc": float(roc_auc_score(y_true, y_prob)),
        "test_pr_auc": float(average_precision_score(y_true, y_prob)),
        "test_accuracy": float(accuracy_score(y_true, y_pred)),
        "test_f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "test_precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "test_recall": float(recall_score(y_true, y_pred, zero_division=0)),
    }


def evaluate_model_on_bundle(model, config, data_bundle, threshold, model_family, unit, evaluation_scope):
    loader = make_loader(
        data_bundle["X_test"],
        data_bundle["y_test"],
        batch_size=config["batch_size"],
        shuffle=False,
    )
    y_true, y_prob = predict_proba(model, loader, torch.device("cpu"))
    row = {
        "model_family": model_family,
        "unit": unit,
        "evaluation_scope": evaluation_scope,
        "test_rows": int(len(y_true)),
        "default_rate": float(np.mean(y_true)),
        "threshold": float(threshold),
    }
    row.update(classification_metrics(y_true, y_prob, threshold))
    return row


def client_test_bundle_with_preprocess(client_df, target_col, use_feature_engineering, seed, preprocess, expected_input_dim):
    model_df = add_features(client_df) if use_feature_engineering else client_df.copy()
    X_raw = model_df.drop(columns=["ID", target_col])
    y_raw = model_df[target_col].astype(np.float32)

    _, X_temp_raw, _, y_temp = train_test_split(
        X_raw, y_raw, test_size=0.30, random_state=seed, stratify=y_raw
    )
    _, X_test_raw, _, y_test = train_test_split(
        X_temp_raw, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
    )

    X_test = preprocess.transform(X_test_raw).astype(np.float32)
    if X_test.shape[1] != expected_input_dim:
        raise ValueError(
            f"Client feature width {X_test.shape[1]} does not match global model input_dim {expected_input_dim}."
        )
    return {"X_test": X_test, "y_test": y_test}


def discover_client_files(client_data_dir):
    client_data_dir = Path(client_data_dir)
    nested = sorted(client_data_dir.glob("client_*/client.csv"))
    if nested:
        return nested
    return sorted(client_data_dir.glob("client_*.csv"))


def client_name_from_path(client_csv):
    client_csv = Path(client_csv)
    return client_csv.parent.name if client_csv.name == "client.csv" else client_csv.stem


def weighted_metric_row(df, model_family, unit="weighted_mean_clients"):
    weights = df["test_rows"].astype(float)
    row = {
        "model_family": model_family,
        "unit": unit,
        "evaluation_scope": "client_test_weighted_average",
        "test_rows": int(weights.sum()),
        "default_rate": np.average(df["default_rate"], weights=weights) if "default_rate" in df else np.nan,
        "threshold": np.nan,
    }
    for col in METRIC_COLS:
        row[col] = float(np.average(df[col], weights=weights))
    return row


def make_latex_table(df, path, float_format="%.3f"):
    latex = df.to_latex(index=False, escape=True, float_format=float_format)
    Path(path).write_text(latex)
    return path


def paper_metric_view(df):
    columns = ["model_family", "unit", "evaluation_scope", "test_rows"] + METRIC_COLS
    out = df[columns].copy()
    out = out.rename(columns={"model_family": "Model", "unit": "Unit", "evaluation_scope": "Scope", "test_rows": "Test N", **METRIC_LABELS})
    return out


## Global Model Evaluation

The global model is evaluated once on the global test split and once on each client test split. The per-client evaluation is useful because it creates a direct paired comparison against each local model.

In [ ]:
global_artifact = load_torch_artifact(GLOBAL_MODEL_PATH)
global_model, global_config = build_mlp_from_artifact(global_artifact)
global_feature_set = global_artifact.get("feature_set", "engineered")
global_threshold = float(global_artifact["best_threshold"])

full_df = read_credit_dataset(DEFAULT_INPUT_PATH)
full_feature_sets = prepare_feature_sets(full_df, target_col=TARGET_COL, seed=SEED)
global_preprocess = full_feature_sets[global_feature_set]["preprocess"]
global_overall_row = evaluate_model_on_bundle(
    global_model,
    global_config,
    full_feature_sets[global_feature_set],
    global_threshold,
    model_family="Global MLP",
    unit="global_test",
    evaluation_scope="global_test",
)

global_client_rows = []
for client_csv in discover_client_files(CLIENT_DATA_DIR):
    client_df = pd.read_csv(client_csv)
    client_test_bundle = client_test_bundle_with_preprocess(
        client_df,
        target_col=TARGET_COL,
        use_feature_engineering=global_artifact.get("use_feature_engineering", global_feature_set == "engineered"),
        seed=SEED,
        preprocess=global_preprocess,
        expected_input_dim=global_artifact["input_dim"],
    )
    global_client_rows.append(
        evaluate_model_on_bundle(
            global_model,
            global_config,
            client_test_bundle,
            global_threshold,
            model_family="Global MLP",
            unit=client_name_from_path(client_csv),
            evaluation_scope="client_test",
        )
    )

global_overall_df = pd.DataFrame([global_overall_row])
global_client_df = pd.DataFrame(global_client_rows)
global_client_weighted_df = pd.DataFrame([weighted_metric_row(global_client_df, "Global MLP")])

pd.concat([global_overall_df, global_client_weighted_df, global_client_df], ignore_index=True)


## Local Model Results

The local model notebook already writes one summary row per client. This section loads those rows and computes a weighted local average.

In [ ]:
local_summary = pd.read_csv(LOCAL_SUMMARY_PATH)
local_client_df = local_summary.copy()
local_client_df["model_family"] = "Local MLP"
local_client_df["unit"] = local_client_df["client"]
local_client_df["evaluation_scope"] = "client_test"
local_client_df["threshold"] = local_client_df["best_threshold"]

local_client_df = local_client_df[
    ["model_family", "unit", "evaluation_scope", "test_rows", "default_rate", "threshold"] + METRIC_COLS
]
local_client_weighted_df = pd.DataFrame([weighted_metric_row(local_client_df, "Local MLP")])

pd.concat([local_client_weighted_df, local_client_df], ignore_index=True)


## Future Federated Results

When the federated-learning code is ready, save a summary CSV with columns like `test_roc_auc`, `test_pr_auc`, `test_f1`, `test_precision`, `test_recall`, `test_accuracy`, and `test_rows`. If the filename matches `federated*_summary.csv` somewhere under `models/`, this notebook will load it automatically. You can also add paths manually in `FUTURE_FEDERATED_SUMMARY_PATHS`.

In [ ]:
federated_frames = []
for path in FEDERATED_SUMMARY_PATHS:
    if not Path(path).exists():
        continue
    fed = pd.read_csv(path)
    fed = fed.copy()
    fed["model_family"] = fed.get("model_family", f"Federated ({Path(path).parent.name})")
    fed["unit"] = fed.get("unit", fed.get("client", "federated"))
    fed["evaluation_scope"] = fed.get("evaluation_scope", "client_test")
    if "threshold" not in fed and "best_threshold" in fed:
        fed["threshold"] = fed["best_threshold"]
    if "default_rate" not in fed:
        fed["default_rate"] = np.nan
    federated_frames.append(fed[["model_family", "unit", "evaluation_scope", "test_rows", "default_rate", "threshold"] + METRIC_COLS])

federated_df = pd.concat(federated_frames, ignore_index=True) if federated_frames else pd.DataFrame(columns=["model_family", "unit", "evaluation_scope", "test_rows", "default_rate", "threshold"] + METRIC_COLS)
federated_df


## Paper-Ready Tables

The first table is the compact headline table. The weighted client averages are usually the easiest to discuss because both global and local models are evaluated across the same client test partitions.

In [ ]:
comparison_df = pd.concat(
    [global_overall_df, global_client_weighted_df, global_client_df, local_client_weighted_df, local_client_df, federated_df],
    ignore_index=True,
)

headline_df = comparison_df[
    comparison_df["evaluation_scope"].isin(["global_test", "client_test_weighted_average"])
].copy()
headline_table = paper_metric_view(headline_df).round(3)

comparison_csv = OUTPUT_DIR / "model_comparison_all_rows.csv"
headline_csv = OUTPUT_DIR / "model_comparison_headline.csv"
headline_tex = OUTPUT_DIR / "model_comparison_headline.tex"

comparison_df.to_csv(comparison_csv, index=False)
headline_table.to_csv(headline_csv, index=False)
make_latex_table(headline_table, headline_tex)

print("Saved", comparison_csv)
print("Saved", headline_csv)
print("Saved", headline_tex)
headline_table


## Per-Client Paired Deltas

This table compares local and global models on the same client test splits. Values are `Local MLP - Global MLP`; positive values favor the local model.

In [ ]:
paired = local_client_df.merge(
    global_client_df,
    on="unit",
    suffixes=("_local", "_global"),
)

delta_rows = []
for _, row in paired.iterrows():
    delta = {
        "client": row["unit"],
        "test_rows": int(row["test_rows_local"]),
        "default_rate": row["default_rate_local"],
    }
    for col in METRIC_COLS:
        delta[f"delta_{col}"] = row[f"{col}_local"] - row[f"{col}_global"]
    delta_rows.append(delta)

delta_df = pd.DataFrame(delta_rows)
delta_table = delta_df.rename(columns={f"delta_{col}": f"Delta {label}" for col, label in METRIC_LABELS.items()}).round(3)

delta_csv = OUTPUT_DIR / "local_minus_global_client_deltas.csv"
delta_tex = OUTPUT_DIR / "local_minus_global_client_deltas.tex"
delta_table.to_csv(delta_csv, index=False)
make_latex_table(delta_table, delta_tex)

print("Saved", delta_csv)
print("Saved", delta_tex)
delta_table


## Client Variability

Mean and standard deviation across clients are useful for describing whether a model family is stable across heterogeneous client distributions.

In [ ]:
client_only = comparison_df[comparison_df["evaluation_scope"] == "client_test"].copy()
variability = (
    client_only
    .groupby("model_family")[METRIC_COLS]
    .agg(["mean", "std"])
)
variability.columns = [f"{METRIC_LABELS[col]} {stat}" for col, stat in variability.columns]
variability = variability.reset_index().rename(columns={"model_family": "Model"}).round(3)

variability_csv = OUTPUT_DIR / "client_metric_variability.csv"
variability_tex = OUTPUT_DIR / "client_metric_variability.tex"
variability.to_csv(variability_csv, index=False)
make_latex_table(variability, variability_tex)

print("Saved", variability_csv)
print("Saved", variability_tex)
variability


## Figures

The following plots are intentionally simple: grouped bars are easy to read and survive well in paper PDFs.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

plot_df = client_only[client_only["model_family"].isin(["Global MLP", "Local MLP"])].copy()
for metric in ["test_roc_auc", "test_pr_auc", "test_f1", "test_recall"]:
    pivot = plot_df.pivot(index="unit", columns="model_family", values=metric)
    ax = pivot.plot(kind="bar", figsize=(8, 4), width=0.75)
    ax.set_title(f"{METRIC_LABELS[metric]} by client")
    ax.set_xlabel("Client")
    ax.set_ylabel(METRIC_LABELS[metric])
    ax.set_ylim(0, 1)
    ax.legend(title="Model")
    fig = ax.get_figure()
    fig.tight_layout()
    output_path = OUTPUT_DIR / f"client_{metric}.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print("Saved", output_path)
    plt.show()


## LaTeX Integration

Use `\input{...}` for the exported `.tex` tables, or copy the generated table body into your paper. For figures, include the PNGs with `\includegraphics`.

Example:

```latex
\begin{table}[ht]
  \centering
  \caption{Comparison of global and local MLP models.}
  \input{reports/model_comparison/model_comparison_headline.tex}
\end{table}

\begin{figure}[ht]
  \centering
  \includegraphics[width=0.8\linewidth]{reports/model_comparison/client_test_f1.png}
  \caption{Client-wise F1 scores for global and local MLP models.}
\end{figure}
```